# PART 2

Haikal Ali_1103223071_TK46GAB

# Pre-Model Workflow dan Data Preprocessing

Notebook ini berisi ringkasan dan contoh kode dari materi **Pre-Model Workflow and Data Preprocessing**.

Materi utama:
- Dampak data mentah terhadap performa model
- Penanganan missing data
- Teknik scaling
- Encoding variabel kategorikal
- Pipeline di scikit-learn
- Feature engineering

## Technical Requirements

Materi menyarankan penggunaan environment Python yang terisolasi agar pekerjaan aman dari instalasi atau library Python lain.

Kebutuhan teknis:
- Git >= 2.46.x
- Python >= 3.9.x
- Repository GitHub sudah di-clone
- Environment Python dibuat dari file `requirements.txt`

## Dampak Data Mentah terhadap Performa Model

Algoritma machine learning belajar dari pola di dalam data. Jika data input bermasalah, seperti memiliki nilai kosong, outlier, bias, noise, atau fitur yang tidak relevan, kemampuan model untuk melakukan generalisasi pada data baru akan menurun.

Prinsip penting pada bagian ini adalah **garbage in, garbage out**. Jika data yang masuk buruk, hasil model juga cenderung buruk.

Masalah umum pada kualitas data:
- **Missing data**: data tidak lengkap karena kesalahan input, kegagalan sistem, atau korupsi data.
- **Outliers**: nilai ekstrem yang dapat mengganggu analisis statistik dan hasil model.
- **Categorical variables**: data kategori harus diubah menjadi format numerik agar dapat diproses model.
- **Feature scaling**: fitur dengan skala berbeda dapat mengganggu model tertentu, terutama model berbasis jarak.
- **Data leakage**: informasi dari data test masuk ke proses training sehingga evaluasi model menjadi terlalu optimistis.

scikit-learn menyediakan beberapa alat preprocessing seperti:
- `Pipeline()`
- `ColumnTransformer()`
- `sklearn.feature_selection`

# Handling Missing Data

Missing data dapat muncul karena human error, kegagalan teknis, atau data corruption. Banyak algoritma machine learning tidak bisa langsung bekerja jika data masih memiliki nilai kosong.

Pada bagian ini dibuat dataset contoh berisi data numerik acak dengan 10 fitur dan beberapa nilai kosong.

## Membuat Dataset dengan Missing Values

In [1]:
import numpy as np
import pandas as pd

np.random.seed(2024)
n_samples = 20
n_features = 10

data = {
  f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
  for i in range(n_features)
}

df = pd.DataFrame(data)

for column in df.columns:
  mask = np.random.random(n_samples) < 0.2
  df.loc[mask, column] = np.nan

display(df)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,NaN,22.966899,21.049022,57.358544,14.302591
7,NaN,NaN,89.538184,69.451294,NaN,47.885551,NaN,33.620401,99.685711,6.683138
8,47.384570,NaN,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,NaN,31.142866,28.865545,NaN,41.004459,41.460336,50.236236,NaN


## SimpleImputer

`SimpleImputer()` adalah metode sederhana untuk menangani missing values. Nilai kosong dapat diganti menggunakan statistik tertentu seperti mean, median, atau most frequent value.

In [2]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)

imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,57.822964,22.966899,21.049022,57.358544,14.302591
7,52.558605,53.864661,89.538184,69.451294,46.055883,47.885551,51.145011,33.620401,99.685711,6.683138
8,47.384570,53.864661,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.897507,31.142866,28.865545,57.822964,41.004459,41.460336,50.236236,48.400044


## KNNImputer

`KNNImputer()` mengisi missing values berdasarkan nilai dari sampel tetangga terdekat di ruang fitur. Jumlah tetangga ditentukan melalui parameter `n_neighbors`.

In [3]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)

knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(
  knn_imputed_data,
  columns=df.columns
)

knn_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,40.482911,22.966899,21.049022,57.358544,14.302591
7,47.629715,77.843853,89.538184,69.451294,32.753328,47.885551,55.683434,33.620401,99.685711,6.683138
8,47.384570,35.180639,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,29.188271,31.142866,28.865545,59.674651,41.004459,41.460336,50.236236,35.000820


## IterativeImputer

`IterativeImputer()` adalah pendekatan yang lebih maju. Metode ini memodelkan setiap fitur yang memiliki missing values sebagai fungsi dari fitur lain secara bergiliran. Missing values diprediksi menggunakan model regresi berdasarkan fitur-fitur lain.

In [4]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer()

iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(
  iterative_imputed_data,
  columns=df.columns
)

iterative_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,55.923578,22.966899,21.049022,57.358544,14.302591
7,52.558131,53.942885,89.538184,69.451294,46.070477,47.885551,50.969808,33.620401,99.685711,6.683138
8,47.384570,54.204978,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.882614,31.142866,28.865545,55.978300,41.004459,41.460336,50.236236,48.429901


## Pemilihan Strategi Imputasi

Pemilihan metode imputasi bergantung pada:
- Tipe data
- Distribusi data
- Pola missing data
- Jumlah missing data

Catatan dari materi:
- Untuk missing data lebih dari 5%, `SimpleImputer()` bisa menjadi pilihan karena ringan.
- Untuk missing data sekitar 5–10%, `KNNImputer()` cocok untuk data numerik.
- `IterativeImputer()` dapat digunakan untuk mempertimbangkan hubungan multivariat.
- Pada dataset yang sangat sparse, fitur dengan missing values tinggi bisa dipertimbangkan untuk dihapus jika tidak penting.
- Beberapa model seperti decision tree dan random forest dapat menangani missing values tanpa imputasi pada kondisi tertentu.

## Latihan Missing Data

Alur umum:
1. Load dataset dan identifikasi missing values.
2. Pilih metode imputasi yang sesuai.
3. Terapkan imputer untuk mengisi missing values.
4. Evaluasi performa model menggunakan cross-validation untuk melihat dampak strategi imputasi.

# Scaling Techniques

Fitur dalam dataset dapat memiliki skala yang sangat berbeda. Contohnya, umur berada pada rentang 0–100, sedangkan pendapatan bisa berada pada rentang 0–100.000.

Beberapa algoritma seperti KNN dan metode berbasis gradient descent sensitif terhadap perbedaan skala. Scaling membantu agar tidak ada satu fitur yang terlalu mendominasi proses pembelajaran.

Dua konsep penting:
- **Standardization**: mengubah data agar memiliki mean 0 dan standard deviation 1.
- **Normalization**: mengubah rentang data agar berada pada skala tertentu, biasanya 0 sampai 1.

## StandardScaler

`StandardScaler()` melakukan standardisasi sehingga fitur memiliki mean 0 dan standard deviation 1. Hasil transformasi berupa Z-score.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(
  scaled_data,
  columns=iterative_imputed_df.columns
)

scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904
6,0.878636,1.094467,-0.091017,0.042422,-1.929442,-0.075466,-1.302124,-1.273574,-0.110399,-1.133636
7,-0.000135,0.001649,1.255789,0.639061,0.000428,-0.388327,-0.007723,-0.734020,1.640810,-1.386962
8,-0.225584,0.012729,-0.935710,1.210940,1.323678,0.147955,-1.015275,0.653400,0.040472,-0.476999
9,-0.336923,-0.665377,-0.000435,-1.050256,-0.832163,-0.073336,-0.468360,-0.397536,-0.405072,0.001002


## MinMaxScaler

`MinMaxScaler()` mengubah nilai fitur ke rentang tertentu, biasanya 0 sampai 1. Teknik ini berguna ketika ingin mempertahankan hubungan antar nilai, tetapi tetap membuat semua fitur berada pada skala yang sebanding.

In [6]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

minmax_scaled_data = minmax_scaler.fit_transform(
  iterative_imputed_df
)
minmax_scaled_df = pd.DataFrame(
  minmax_scaled_data,
  columns=iterative_imputed_df.columns
)

minmax_scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710
6,0.751199,0.811657,0.468490,0.611621,0.000000,0.524114,0.070723,0.218016,0.567681,0.107208
7,0.537276,0.512946,0.888472,0.775311,0.518428,0.431454,0.432317,0.353891,1.000000,0.027004
8,0.482394,0.515975,0.205085,0.932208,0.873897,0.590284,0.150855,0.703283,0.604927,0.315101
9,0.455290,0.330621,0.496737,0.311840,0.294766,0.524745,0.303637,0.438627,0.494936,0.466437


## Normalizer

`Normalizer()` melakukan normalisasi pada setiap sampel agar memiliki unit norm. Teknik ini berguna pada sparse data atau ketika setiap sampel ingin diperlakukan setara tanpa memperhatikan besar magnitudonya.

In [7]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer()

normalized_data = normalizer.fit_transform(
  iterative_imputed_df
)
normalized_df = pd.DataFrame(
  normalized_data,
  columns=iterative_imputed_df.columns
)

normalized_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177
6,0.460521,0.505281,0.318139,0.354119,0.039204,0.354133,0.145437,0.133292,0.363220,0.090570
7,0.274582,0.281816,0.467778,0.362837,0.240688,0.250170,0.266284,0.175644,0.520792,0.034915
8,0.265283,0.303467,0.143277,0.461427,0.411012,0.345225,0.163323,0.369203,0.341538,0.190645
9,0.321287,0.273524,0.379002,0.223196,0.206875,0.401188,0.293873,0.297140,0.360036,0.347090


## Dampak Scaling terhadap Model

Scaling tidak selalu dibutuhkan. Model berbasis tree seperti decision tree dan random forest bekerja menggunakan nilai mentah fitur.

Dampak scaling pada beberapa model:
- **KNN**: sensitif terhadap distance metric.
- **SVM**: sensitif terhadap besar nilai fitur.
- **Linear regression**: standardisasi dapat membantu proses konvergensi.

## Latihan Scaling

Alur umum:
1. Load dataset dan identifikasi fitur yang perlu scaling.
2. Pilih metode scaling berdasarkan karakteristik data.
3. Terapkan teknik scaling.
4. Latih beberapa model menggunakan dataset hasil scaling.
5. Evaluasi performa model menggunakan metrik seperti accuracy, precision, recall, atau mean squared error.

# Encoding Categorical Variables

Variabel kategorikal berisi nilai diskrit seperti kategori, label, atau grup. Sebagian besar algoritma machine learning membutuhkan input numerik, sehingga variabel kategorikal harus diubah ke bentuk numerik.

Tipe variabel kategorikal:
- **Nominal variables**: kategori tanpa urutan, misalnya warna atau brand.
- **Ordinal variables**: kategori yang memiliki urutan, misalnya rating 1 sampai 5.

## Membuat Dataset Kategorikal

In [8]:
import numpy as np

np.random.seed(2024)
categories = ["A", "B", "C", "D"]

categorical_data = pd.DataFrame(
  {
    "Department": np.random.choice(
      categories,
      size=20
    ),
    "Position": np.random.choice(
      ["Junior", "Senior", "Manager"],
      size=20
    ),
    "Location": np.random.choice(
      ["NY", "SF", "LA", "CHI"],
      size=20
    ),
  }
)

display(categorical_data)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


## OneHotEncoder

`OneHotEncoder()` mengubah variabel kategorikal nominal menjadi beberapa kolom biner. Setiap kategori menjadi fitur baru, dengan nilai 1 jika baris tersebut memiliki kategori tersebut dan 0 jika tidak.

In [9]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)

onehot_encoded_data = onehot_encoder.fit_transform(
  categorical_data
)
onehot_encoded_df = pd.DataFrame(
  onehot_encoded_data,
  columns=onehot_encoder.get_feature_names_out()
)

onehot_encoded_df

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


Catatan: jika dataset memiliki banyak fitur kategorikal atau banyak kategori, `OneHotEncoder()` dapat menghasilkan dataset yang sangat besar dan sparse.

## LabelEncoder

`LabelEncoder()` memberikan angka unik untuk setiap kategori. Teknik ini sederhana, tetapi dapat membuat model salah menganggap angka tersebut sebagai urutan atau jarak antar kategori.

In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

label_encoded_df = pd.DataFrame()

for column in categorical_data.columns:
  label_encoded_df[f"{column}_encoded"] = (
    label_encoder.fit_transform(
      categorical_data[column]
    )
  )

label_encoded_df

,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1
6,2,1,3
7,3,1,1
8,1,0,1
9,3,1,0


Catatan: `LabelEncoder()` memberikan nilai secara arbitrer pada fitur kategorikal. Sebelum menggunakannya, perlu dipertimbangkan apakah fitur tersebut nominal atau ordinal.

## ColumnTransformer

`ColumnTransformer()` digunakan ketika dataset memiliki campuran fitur numerik dan kategorikal. Dengan alat ini, preprocessing berbeda dapat diterapkan pada kolom yang berbeda.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

np.random.seed(2024)

mixed_data = pd.DataFrame(
  {
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(
      np.random.normal(60000, 15000, size=20),
      2
    ),
    "Experience": np.random.randint(
      1,
      20,
      size=20
    ),
    "Department": np.random.choice(
      ["IT", "HR", "Sales", "Finance"],
      size=20
    ),
    "Position": np.random.choice(
      ["Junior", "Senior", "Manager"],
      size=20
    ),
  }
)

display(mixed_data)

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior
6,60,49585.62,14,Finance,Junior
7,35,47992.58,17,HR,Manager
8,27,54833.02,9,Sales,Senior
9,57,93535.56,14,Sales,Senior


In [12]:
column_transformer = ColumnTransformer(
  transformers=[
    (
      "num",
      StandardScaler(),
      ["Age", "Salary", "Experience"]
    ),
    (
      "cat",
      OneHotEncoder(),
      ["Department", "Position"]
    ),
  ],
  remainder="passthrough",
)

transformed_data = column_transformer.fit_transform(mixed_data)

numeric_cols = [
  "Age_scaled",
  "Salary_scaled",
  "Experience_scaled"
]

categorical_cols = (
  column_transformer
  .named_transformers_["cat"]
  .get_feature_names_out(
    ["Department", "Position"]
  )
)

transformed_df = pd.DataFrame(
  transformed_data,
  columns=numeric_cols + list(categorical_cols)
)

transformed_df

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358300,-1.577167,-0.607848,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023186,-0.565241,-0.420818,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562482,0.852382,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6,0.949309,-0.895798,0.701364,1.0,0.0,0.0,0.0,1.0,0.0,0.0
7,-0.897596,-0.991489,1.262454,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8,-1.488606,-0.580596,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.727681,1.744195,0.701364,0.0,0.0,0.0,1.0,0.0,0.0,1.0


## Latihan Encoding

Alur umum:
1. Identifikasi fitur kategorikal yang perlu encoding.
2. Pilih metode encoding berdasarkan tipe variabel, nominal atau ordinal.
3. Terapkan encoding menggunakan preprocessing tools dari scikit-learn.
4. Gabungkan kembali fitur hasil encoding dengan fitur numerik.
5. Evaluasi performa model untuk memahami dampak metode encoding.

# Introduction to Pipelines in scikit-learn

`Pipeline()` digunakan untuk menggabungkan beberapa langkah preprocessing dan training model dalam satu objek. Pipeline membantu membuat workflow lebih rapi, konsisten, dan mengurangi risiko error.

Keuntungan pipeline:
- Eksekusi langkah secara berurutan.
- Kode lebih sederhana.
- Transformasi konsisten pada data training dan testing.
- Mengurangi risiko data leakage.
- Lebih mudah digunakan bersama hyperparameter tuning.
- Mendukung modularitas.

## Struktur Umum Pipeline

```python
pipeline = Pipeline(
  [
    ("name of step", transformer),
    ("name of step", transformer),
    ...,
    ("name of step", estimator)
  ]
)
```

Langkah terakhir pipeline dapat berupa estimator seperti `LogisticRegression()`.

## Membuat Pipeline untuk Imputasi dan Scaling

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = transformed_df.iloc[:, :-1]
y = transformed_df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.2,
  random_state=2024
)

In [14]:
pipeline = Pipeline([
  (
    "imputer",
    SimpleImputer(strategy="mean")
  ),
  (
    "scaler",
    StandardScaler()
  ),
])

X_train_transformed = pipeline.fit_transform(X_train)

X_test_transformed = pipeline.transform(X_test)

X_train_transformed = pd.DataFrame(
  X_train_transformed,
  columns=X_train.columns,
  index=X_train.index
)

X_test_transformed = pd.DataFrame(
  X_test_transformed,
  columns=X_test.columns,
  index=X_test.index
)

X_train_transformed

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
13,0.834298,-0.867451,-1.559779,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,-1.0
12,-0.689202,1.282818,-0.150946,-0.577350,2.081666,-0.480384,-0.774597,-0.577350,1.0
16,0.544107,-1.173357,-1.761041,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,1.0
5,-1.342131,0.821525,-0.150946,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0
17,1.414679,0.611115,0.654101,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,1.0
2,-1.414679,-1.479801,1.257887,1.732051,-0.480384,-0.480384,-0.774597,-0.577350,1.0
10,1.269583,0.576819,0.452839,1.732051,-0.480384,-0.480384,-0.774597,1.732051,-1.0
3,0.544107,-1.475450,-0.553470,-0.577350,-0.480384,2.081666,-0.774597,1.732051,-1.0
1,0.906845,1.060444,1.459148,-0.577350,-0.480384,-0.480384,1.290994,1.732051,-1.0
9,0.906845,1.664674,0.855363,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0


## Pentingnya Split Sebelum Transformasi Data

`train_test_split()` perlu dilakukan sebelum transformasi data untuk menjaga integritas workflow machine learning.

Alasannya:
- Menghindari data leakage.
- Evaluasi model menjadi lebih realistis.
- Parameter transformasi, seperti mean dan standard deviation, hanya dihitung dari data training.
- Data test ditransformasikan menggunakan parameter yang sama dari data training.
- Pipeline mengurangi kompleksitas kode dan risiko error.

## Visualisasi Pipeline

In [15]:
from sklearn import set_config

set_config(display="diagram")

pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

## Latihan Pipeline

Alur umum:
1. Tentukan preprocessing yang dibutuhkan dataset.
2. Pilih algoritma machine learning yang sesuai.
3. Gunakan `Pipeline()` untuk menggabungkan preprocessing dan training model.
4. Latih pipeline pada data training.
5. Gunakan pipeline untuk prediksi pada data baru.
6. Evaluasi performa model dengan metrik yang sesuai.

# Feature Engineering

Feature engineering adalah istilah umum untuk dua aktivitas utama:
1. **Feature extraction**: membuat fitur baru dari data yang sudah ada.
2. **Feature selection**: memilih fitur yang paling relevan dan membuang fitur yang tidak informatif.

Tujuannya adalah meningkatkan performa model, mengurangi noise, dan membuat model lebih sederhana.

## PolynomialFeatures

`PolynomialFeatures()` digunakan untuk menghasilkan fitur polynomial dan fitur interaksi dari fitur numerik yang sudah ada.

In [16]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)

poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
  poly_features,
  columns=poly.get_feature_names_out(
    X_train_transformed.columns
  )
)

poly_features_df

,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_IT^2,Department_IT Department_Sales,Department_IT Position_Junior,Department_IT Position_Manager,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Position_Junior^2,Position_Junior Position_Manager,Position_Manager^2
0,1.0,0.834298,-0.867451,-1.559779,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,-1.0,...,4.333333,-1.612452,-1.201850,-2.081666,0.600000,0.447214,0.774597,0.333333,0.577350,1.0
1,1.0,-0.689202,1.282818,-0.150946,-0.577350,2.081666,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
2,1.0,0.544107,-1.173357,-1.761041,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,1.0,...,4.333333,-1.612452,-1.201850,2.081666,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
3,1.0,-1.342131,0.821525,-0.150946,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0
4,1.0,1.414679,0.611115,0.654101,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,1.0,...,0.230769,-0.620174,0.277350,-0.480384,1.666667,-0.745356,1.290994,0.333333,-0.577350,1.0
5,1.0,-1.414679,-1.479801,1.257887,1.732051,-0.480384,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
6,1.0,1.269583,0.576819,0.452839,1.732051,-0.480384,-0.480384,-0.774597,1.732051,-1.0,...,0.230769,0.372104,-0.832050,0.480384,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
7,1.0,0.544107,-1.475450,-0.553470,-0.577350,-0.480384,2.081666,-0.774597,1.732051,-1.0,...,4.333333,-1.612452,3.605551,-2.081666,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
8,1.0,0.906845,1.060444,1.459148,-0.577350,-0.480384,-0.480384,1.290994,1.732051,-1.0,...,0.230769,-0.620174,-0.832050,0.480384,1.666667,2.236068,-1.290994,3.000000,-1.732051,1.0
9,1.0,0.906845,1.664674,0.855363,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0


## KBinsDiscretizer

`KBinsDiscretizer()` mengubah variabel kontinu menjadi variabel kategorikal dengan membagi nilai ke dalam beberapa bin atau bucket.

Parameter penting:
- `n_bins`: jumlah bucket.
- `encode`: bentuk output.
- `strategy`: cara menentukan batas bucket, misalnya `uniform`, `quantile`, atau `kmeans`.

In [17]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(
  n_bins=3,
  encode="ordinal",
  strategy="uniform"
)

binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(
  binned_data,
  columns=X_train_transformed.columns
)

binned_df

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0
5,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0
6,2.0,1.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0
7,2.0,0.0,1.0,0.0,0.0,2.0,0.0,2.0,0.0
8,2.0,2.0,2.0,0.0,0.0,0.0,2.0,2.0,0.0
9,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0


## Recursive Feature Elimination

`RFE()` menghapus fitur yang dianggap paling tidak penting secara rekursif berdasarkan estimator tertentu. Model dilatih menggunakan semua fitur, fitur paling tidak penting dibuang, lalu model dilatih ulang.

In [18]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(
  estimator=LinearRegression(),
  n_features_to_select=1
)

rfe.fit(X_train_transformed, y_train)

rfe.ranking_

array([5, 3, 8, 7, 6, 4, 9, 2, 1])

## SelectFromModel

`SelectFromModel()` memilih fitur berdasarkan bobot atau importance dari model tertentu. Metode ini berguna untuk memilih fitur yang paling informatif.

In [19]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

selector = SelectFromModel(
  estimator=LinearRegression(),
  prefit=False,
  threshold="mean"
)

selector.fit(X_train_transformed, y_train)

selected_features_mask = selector.get_support()

selected_features = X_train_transformed.columns[
  selected_features_mask
].tolist()

selected_features

['Position_Junior', 'Position_Manager']

In [20]:
feature_importance = pd.DataFrame({
  "Feature": X_train_transformed.columns,
  "Importance": selector.estimator_.coef_,
  "Selected": selected_features_mask
})

feature_importance.sort_values(
  "Importance",
  key=abs,
  ascending=False
)

,Feature,Importance,Selected
8,Position_Manager,-5.000000e-01,True
7,Position_Junior,-4.330127e-01,True
2,Experience_scaled,-1.191755e-15,False
1,Salary_scaled,9.298118e-16,False
0,Age_scaled,6.755007e-16,False
3,Department_Finance,5.342948e-16,False
4,Department_HR,-2.983724e-16,False
5,Department_IT,-2.810252e-16,False
6,Department_Sales,-1.561251e-16,False


## Latihan Feature Engineering

Alur umum:
1. Pahami karakteristik dataset dan cari area yang bisa dibuatkan fitur baru.
2. Gunakan transformasi matematika atau domain knowledge untuk membuat variabel baru.
3. Terapkan teknik seperti `RFE()` atau `SelectFromModel()` untuk mempertahankan fitur informatif.
4. Latih model menggunakan dataset asli dan dataset hasil feature engineering.
5. Bandingkan performa model.

# Practical Exercises on Data Preprocessing

Bagian ini meminta penggunaan dataset **California Housing** yang tersedia di scikit-learn. Dataset memiliki 20.640 record dan 9 fitur. Target yang diprediksi adalah rata-rata harga rumah per 100.000 rumah.

Tugas:
1. Load dataset California Housing.
2. Split data.
3. Buat pipeline komprehensif dengan minimal tiga langkah, termasuk estimator sebagai langkah terakhir.
4. Fit pipeline ke dataset.
5. Evaluasi performa pipeline pada data test.

Materi juga menampilkan decision guide untuk memilih teknik transformasi data berdasarkan karakteristik data:
- Jika ada outlier, gunakan `RobustScaler`, `QuantileTransformer`, atau transformasi power.
- Jika distribusi normal, gunakan `StandardScaler`.
- Jika distribusi skewed, gunakan transformasi seperti Yeo-Johnson atau Box-Cox.
- Jika ada kebutuhan khusus seperti norm L1/L2, gunakan `Normalizer`.
- Jika ingin rentang custom, gunakan `MinMaxScaler(feature_range)`.